In [1]:

from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Sequence
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

try:
    from sqlalchemy import create_engine
except Exception:
    create_engine = None

try:
    import psycopg2
except Exception:
    psycopg2 = None

plt.switch_backend("Agg")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Project helpers and database connection

In [2]:
ID_LIKE_COLUMNS = {
    "year",
    "tract_geoid",
    "tract_name",
    "statefp",
    "countyfp",
    "geoid",
    "geo_id",
    "name",
    "tract_id",
    "tract",
    "is_stable_all_4_years",
    "is_stable",
}

DEFAULT_PRIORITY_PATTERNS: Dict[str, List[str]] = {
    "median_household_income": [
        r"median.*household.*income",
        r"household.*income.*median",
        r"b19013",
    ],
    "poverty_rate": [
        r"poverty.*rate",
        r"below.*poverty",
        r"pct.*poverty",
        r"s1701",
    ],
    "rent_burden": [
        r"rent.*burden",
        r"gross.*rent.*income",
        r"b25070",
    ],
    "unemployment_rate": [
        r"unemploy",
        r"s2301",
    ],
    "education_bachelors_or_higher": [
        r"bachelor",
        r"graduate",
        r"college",
        r"s1501",
    ],
    "owner_occupied_share": [
        r"owner.*occupied",
        r"owner",
        r"b25003",
    ],
    "median_gross_rent": [
        r"median.*gross.*rent",
        r"gross.*rent.*median",
    ],
    "median_home_value": [
        r"median.*value",
        r"home.*value",
    ],
    "limited_english_share": [
        r"limited.*english",
        r"english.*less",
        r"s1601",
    ],
    "population_65_plus_share": [
        r"65.*over",
        r"65.*plus",
        r"elder",
        r"senior",
    ],
}

class DatabaseError(RuntimeError):
    pass


def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".env").exists():
            return candidate
        if (candidate / "scripts").exists() and (candidate / "outputs").exists():
            return candidate
    return Path.cwd()


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")


def get_db_uri() -> str:
    required = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
    missing = [key for key in required if not os.getenv(key)]
    if missing:
        raise DatabaseError(f"Missing required environment variables: {', '.join(missing)}")

    host = os.getenv("DB_HOST")
    port = os.getenv("DB_PORT")
    name = os.getenv("DB_NAME")
    user = quote_plus(os.getenv("DB_USER", ""))
    password = quote_plus(os.getenv("DB_PASSWORD", ""))
    return f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{name}"


def get_sqlalchemy_engine():
    if create_engine is None:
        raise DatabaseError("sqlalchemy is not installed. Install: pip install sqlalchemy psycopg2-binary")
    return create_engine(get_db_uri())


def get_psycopg_connection():
    if psycopg2 is None:
        raise DatabaseError("psycopg2 is not installed. Install: pip install psycopg2-binary")
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
    )


def read_sql(sql: str, params: Optional[dict] = None) -> pd.DataFrame:
    if create_engine is not None:
        engine = get_sqlalchemy_engine()
        with engine.connect() as conn:
            return pd.read_sql_query(sql, conn, params=params)

    conn = get_psycopg_connection()
    try:
        return pd.read_sql_query(sql, conn, params=params)
    finally:
        conn.close()


def resolve_table_name(default_table: str = "fact_acs_tract_profile") -> str:
    schema = os.getenv("DB_SCHEMA", "public")
    if "." in default_table:
        return default_table
    return f"{schema}.{default_table}"


def get_output_root() -> Path:
    return PROJECT_ROOT / "outputs" / "acs" / "analysis" / "eda"


def ensure_dirs(*paths: Path) -> None:
    for path in paths:
        path.mkdir(parents=True, exist_ok=True)


def sanitize_filename(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")


def save_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def get_years(df: pd.DataFrame) -> List[int]:
    return sorted(pd.Series(df["year"]).dropna().astype(int).unique().tolist())


def maybe_cast_numeric(df: pd.DataFrame, exclude: Optional[Sequence[str]] = None) -> pd.DataFrame:
    exclude = set(exclude or [])
    out = df.copy()
    for col in out.columns:
        if col in exclude:
            continue
        if out[col].dtype == object:
            converted = pd.to_numeric(out[col], errors="coerce")
            if converted.notna().sum() >= max(3, int(0.6 * len(out))):
                out[col] = converted
    return out


def identify_numeric_columns(df: pd.DataFrame) -> List[str]:
    numeric_cols = []
    for col in df.columns:
        if col.lower() in ID_LIKE_COLUMNS:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)
    return numeric_cols


def find_best_metric_matches(
    numeric_columns: Sequence[str],
    pattern_map: Optional[Dict[str, List[str]]] = None,
) -> Dict[str, str]:
    pattern_map = pattern_map or DEFAULT_PRIORITY_PATTERNS
    matches: Dict[str, str] = {}
    lowered = {col: col.lower() for col in numeric_columns}

    for friendly_name, patterns in pattern_map.items():
        for pattern in patterns:
            found = [col for col, low in lowered.items() if re.search(pattern, low)]
            if found:
                matches[friendly_name] = sorted(found, key=len)[0]
                break

    return matches


def pick_top_numeric_columns(df: pd.DataFrame, numeric_columns: List[str], max_columns: int = 12) -> List[str]:
    scored = []
    for col in numeric_columns:
        s = pd.to_numeric(df[col], errors="coerce")
        scored.append(
            {
                "column": col,
                "non_null_count": int(s.notna().sum()),
                "std_dev": float(s.std(skipna=True)) if s.notna().any() else 0.0,
                "n_unique": int(s.nunique(dropna=True)),
            }
        )

    scored_df = pd.DataFrame(scored).sort_values(
        ["non_null_count", "n_unique", "std_dev", "column"],
        ascending=[False, False, False, True],
    )
    return scored_df["column"].head(max_columns).tolist()


def standardize_for_heatmap(df: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for col in columns:
        s = pd.to_numeric(df[col], errors="coerce")
        mean = s.mean(skipna=True)
        std = s.std(skipna=True)
        if pd.isna(std) or std == 0:
            out[col] = 0.0
        else:
            out[col] = (s - mean) / std
    return out


def load_acs_profile(table_name: str = "fact_acs_tract_profile") -> pd.DataFrame:
    full_table = resolve_table_name(table_name)
    sql = f'''
    SELECT *
    FROM {full_table}
    ORDER BY year, tract_geoid
    '''
    return read_sql(sql)

## 2. EDA functions

In [3]:

def make_output_dirs(output_root: Path) -> Dict[str, Path]:
    paths = {
        "data": output_root / "data",
        "summary": output_root / "summary",
        "correlation": output_root / "correlation",
        "rankings": output_root / "rankings",
        "plots_dist": output_root / "plots" / "distributions",
        "plots_trend": output_root / "plots" / "trends",
        "plots_corr": output_root / "plots" / "correlation_heatmaps",
        "plots_heat": output_root / "plots" / "tract_heatmaps",
    }
    ensure_dirs(*paths.values())
    return paths


def export_base_datasets(df: pd.DataFrame, paths: Dict[str, Path]) -> None:
    save_csv(df, paths["data"] / "acs_profile_all_years_full.csv")
    save_csv(df[df["is_stable_all_4_years"] == 1].copy(), paths["data"] / "acs_profile_all_years_stable.csv")


def build_run_summary(df: pd.DataFrame, numeric_columns: List[str], selected_metrics: Dict[str, str]) -> pd.DataFrame:
    years = get_years(df)
    return pd.DataFrame(
        {
            "dataset": ["fact_acs_tract_profile"],
            "row_count": [len(df)],
            "year_count": [len(years)],
            "years": [", ".join(map(str, years))],
            "tract_count_union": [df["tract_geoid"].nunique(dropna=True)],
            "stable_tract_count": [df.loc[df["is_stable_all_4_years"] == 1, "tract_geoid"].nunique(dropna=True)],
            "numeric_column_count": [len(numeric_columns)],
            "selected_metric_count": [len(selected_metrics)],
            "selected_metrics": [", ".join(f"{k}:{v}" for k, v in selected_metrics.items())],
        }
    )


def year_tract_counts(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("year")
        .agg(
            tract_rows=("tract_geoid", "count"),
            distinct_tracts=("tract_geoid", "nunique"),
            stable_tract_rows=("is_stable_all_4_years", lambda s: int((s == 1).sum())),
        )
        .reset_index()
        .sort_values("year")
    )


def numeric_summary_all_years(df: pd.DataFrame, numeric_columns: List[str]) -> pd.DataFrame:
    rows = []
    for year, sub in df.groupby("year"):
        for col in numeric_columns:
            series = pd.to_numeric(sub[col], errors="coerce")
            rows.append(
                {
                    "year": int(year),
                    "metric": col,
                    "non_null_count": int(series.notna().sum()),
                    "missing_count": int(series.isna().sum()),
                    "min_value": series.min(skipna=True),
                    "p25": series.quantile(0.25),
                    "median": series.median(skipna=True),
                    "mean": series.mean(skipna=True),
                    "p75": series.quantile(0.75),
                    "max_value": series.max(skipna=True),
                    "std_dev": series.std(skipna=True),
                }
            )
    return pd.DataFrame(rows).sort_values(["metric", "year"])


def missingness_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for year, sub in df.groupby("year"):
        for col in df.columns:
            nulls = int(sub[col].isna().sum())
            rows.append(
                {
                    "year": int(year),
                    "column_name": col,
                    "null_count": nulls,
                    "null_pct": round((nulls / len(sub)) * 100, 2) if len(sub) else np.nan,
                }
            )
    return pd.DataFrame(rows).sort_values(["year", "null_pct"], ascending=[True, False])


def save_correlations(df: pd.DataFrame, numeric_columns: List[str], paths: Dict[str, Path]) -> None:
    for year, sub in df.groupby("year"):
        corr = sub[numeric_columns].corr(numeric_only=True)
        save_csv(
            corr.reset_index().rename(columns={"index": "metric"}),
            paths["correlation"] / f"correlation_matrix_{year}.csv",
        )

        fig, ax = plt.subplots(figsize=(max(8, len(corr) * 0.6), max(6, len(corr) * 0.5)))
        cax = ax.imshow(corr.values, aspect="auto")
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.index)))
        ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
        ax.set_yticklabels(corr.index, fontsize=8)
        ax.set_title(f"ACS Correlation Heatmap {year}")
        fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout()
        fig.savefig(paths["plots_corr"] / f"correlation_heatmap_{year}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)


def save_rankings(df: pd.DataFrame, metric_map: Dict[str, str], paths: Dict[str, Path]) -> None:
    for friendly_name, column in metric_map.items():
        if column not in df.columns:
            continue

        rows = []
        for year, sub in df.groupby("year"):
            ranked = sub[["year", "tract_geoid", column]].copy()
            if "tract_name" in sub.columns:
                ranked["tract_name"] = sub["tract_name"]

            ranked = ranked.sort_values(column, ascending=False, na_position="last")
            top5 = ranked.head(5).copy()
            top5["rank_group"] = "top_5"
            top5["rank_order"] = range(1, len(top5) + 1)

            bottom5 = ranked.tail(5).copy().sort_values(column, ascending=True, na_position="last")
            bottom5["rank_group"] = "bottom_5"
            bottom5["rank_order"] = range(1, len(bottom5) + 1)

            combined = pd.concat([top5, bottom5], ignore_index=True)
            combined = combined.rename(columns={column: "metric_value"})
            combined["metric_name"] = column
            combined["metric_label"] = friendly_name
            rows.append(combined)

        if rows:
            out = pd.concat(rows, ignore_index=True)
            cols = [
                c for c in
                ["year", "metric_label", "metric_name", "rank_group", "rank_order", "tract_geoid", "tract_name", "metric_value"]
                if c in out.columns
            ]
            save_csv(out[cols], paths["rankings"] / f"rankings_{sanitize_filename(friendly_name)}.csv")


def plot_metric_distributions(df: pd.DataFrame, metric_map: Dict[str, str], paths: Dict[str, Path]) -> None:
    years = get_years(df)
    for friendly_name, column in metric_map.items():
        data = [pd.to_numeric(df.loc[df["year"] == year, column], errors="coerce").dropna() for year in years]
        if not any(len(x) for x in data):
            continue

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.boxplot(data, tick_labels=[str(y) for y in years])
        ax.set_title(f"Distribution by Year: {friendly_name}")
        ax.set_xlabel("Year")
        ax.set_ylabel(column)
        fig.tight_layout()
        fig.savefig(paths["plots_dist"] / f"boxplot_{sanitize_filename(friendly_name)}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)


def plot_metric_trends(df: pd.DataFrame, metric_map: Dict[str, str], paths: Dict[str, Path]) -> None:
    for friendly_name, column in metric_map.items():
        trend = (
            df.groupby("year")[column]
            .agg(median="median", mean="mean")
            .reset_index()
            .sort_values("year")
        )
        if trend.empty:
            continue

        save_csv(
            trend.assign(metric_label=friendly_name, metric_name=column),
            paths["summary"] / f"trend_{sanitize_filename(friendly_name)}.csv",
        )

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(trend["year"], trend["median"], marker="o", label="median")
        ax.plot(trend["year"], trend["mean"], marker="o", label="mean")
        ax.set_title(f"All-Year Trend: {friendly_name}")
        ax.set_xlabel("Year")
        ax.set_ylabel(column)
        ax.legend()
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(paths["plots_trend"] / f"trend_{sanitize_filename(friendly_name)}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)


def build_tract_heatmaps(df: pd.DataFrame, metric_map: Dict[str, str], paths: Dict[str, Path], stable_only: bool = False) -> None:
    subset = df[df["is_stable_all_4_years"] == 1].copy() if stable_only else df.copy()
    title_prefix = "Stable 4-Year Tracts" if stable_only else "All Available Tracts"
    file_prefix = "stable" if stable_only else "full"

    metrics = [col for col in metric_map.values() if col in subset.columns]
    if not metrics or subset.empty:
        return

    standardized = standardize_for_heatmap(subset, metrics)
    plot_df = subset[["year", "tract_geoid"]].copy()
    plot_df[metrics] = standardized
    plot_df["tract_year"] = plot_df["year"].astype(str) + " | " + plot_df["tract_geoid"].astype(str)
    plot_df = plot_df.sort_values(["year", "tract_geoid"])
    matrix = plot_df.set_index("tract_year")[metrics]

    fig_height = min(max(8, len(matrix) * 0.18), 22)
    fig, ax = plt.subplots(figsize=(max(10, len(metrics) * 1.0), fig_height))
    cax = ax.imshow(matrix.values, aspect="auto")
    ax.set_xticks(range(len(metrics)))
    ax.set_xticklabels(metrics, rotation=90, fontsize=8)
    y_ticks = np.linspace(0, len(matrix) - 1, min(len(matrix), 25), dtype=int) if len(matrix) > 0 else []
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([matrix.index[i] for i in y_ticks], fontsize=7)
    ax.set_title(f"Standardized Tract-Metric Heatmap: {title_prefix}")
    fig.colorbar(cax, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(paths["plots_heat"] / f"tract_metric_heatmap_{file_prefix}.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    save_csv(matrix.reset_index(), paths["summary"] / f"tract_metric_heatmap_{file_prefix}.csv")


def save_metric_inventory(metric_map: Dict[str, str], numeric_columns: List[str], paths: Dict[str, Path]) -> None:
    rows = []
    for friendly_name, column in metric_map.items():
        rows.append({"metric_label": friendly_name, "column_name": column, "selection_type": "priority_match"})
    for column in numeric_columns:
        if column not in metric_map.values():
            rows.append({"metric_label": column, "column_name": column, "selection_type": "numeric_column"})
    save_csv(pd.DataFrame(rows), paths["summary"] / "metric_inventory.csv")

## 3. Parameters

Update `FACT_TABLE` only if you are using a different schema-qualified table name.

In [4]:

FACT_TABLE = "fact_acs_tract_profile"
MAX_METRICS = 12

OUTPUT_ROOT = get_output_root()
PATHS = make_output_dirs(OUTPUT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("DB schema:", os.getenv("DB_SCHEMA", "public"))
print("Fact table:", FACT_TABLE)

Project root: D:\Projects\Community-Pulse
Output root: D:\Projects\Community-Pulse\outputs\acs\analysis\eda
DB schema: public
Fact table: fact_acs_tract_profile


## 4. Load ACS profile fact and do basic checks

In [5]:

df = load_acs_profile(table_name=FACT_TABLE)

if df.empty:
    raise RuntimeError("No rows returned from fact_acs_tract_profile. Check the table name and schema.")

if "year" not in df.columns:
    raise RuntimeError("fact_acs_tract_profile must contain a year column.")

if "tract_geoid" not in df.columns:
    raise RuntimeError("fact_acs_tract_profile must contain a tract_geoid column.")

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)

df = maybe_cast_numeric(df, exclude=["tract_geoid", "tract_name", "name"])
numeric_columns = identify_numeric_columns(df)

metric_map = find_best_metric_matches(numeric_columns)

if len(metric_map) < min(6, max(1, len(numeric_columns))):
    extra_candidates = [
        col for col in pick_top_numeric_columns(df, numeric_columns, max_columns=MAX_METRICS)
        if col not in metric_map.values()
    ]
    for i, column in enumerate(extra_candidates, start=1):
        metric_map[f"auto_metric_{i:02d}"] = column

corr_columns = list(
    dict.fromkeys([
        *metric_map.values(),
        *pick_top_numeric_columns(df, numeric_columns, max_columns=MAX_METRICS),
    ])
)

print("Rows:", len(df))
print("Years:", get_years(df))
print("Union tract count:", df["tract_geoid"].nunique(dropna=True))
print("Stable tract count:", df.loc[df["is_stable_all_4_years"] == 1, "tract_geoid"].nunique(dropna=True))
print("Numeric column count:", len(numeric_columns))
print("Selected metric count:", len(metric_map))

Rows: 187
Years: [2019, 2021, 2022, 2023]
Union tract count: 53
Stable tract count: 38
Numeric column count: 42
Selected metric count: 14


In [6]:

display(df.head())
display(pd.DataFrame({"selected_metric_label": list(metric_map.keys()), "column_name": list(metric_map.values())}))

,year,tract_geoid,geo_id,statefp,countyfp,tractce,tract_number,tract_name_canonical,tract_name_latest,county_name,state_name,first_year_seen,last_year_seen,year_count,is_stable_all_4_years,year_label,acs_dataset,acs_period,median_household_income,median_household_income_moe,occupied_units,owner_occupied_units,renter_occupied_units,pct_owner_occupied,pct_renter_occupied,renter_hh_rent_burden_base,rent_30_34,rent_35_39,rent_40_49,rent_50_plus,rent_not_computed,pct_rent_burden_30_plus,pct_rent_burden_50_plus,pct_rent_not_computed,housing_units_total,occupied_housing_units_dp04,vacant_housing_units_dp04,pct_occupied_housing_units,pct_vacant_housing_units,for_rent_units,rented_not_occupied_units,for_sale_only_units,sold_not_occupied_units,seasonal_recreational_units,migrant_worker_units,other_vacant_units,total_population,not_hispanic_total,white_non_hispanic_population,black_non_hispanic_population,hispanic_population,pct_white_non_hispanic,pct_black_non_hispanic,pct_hispanic,has_core_housing_metrics,has_batch2_metrics,has_batch3_metrics,b19013_metrics_json,b25003_metrics_json,b25070_metrics_json,dp04_metrics_json,s1101_metrics_json,s1701_metrics_json,s1901_metrics_json,s2301_metrics_json,s1501_metrics_json,s2401_metrics_json,b03002_metrics_json,s0101_metrics_json,s1601_metrics_json
0,2019,17019000200,1400000US17019000200,17,019,000200,2,Census Tract 2,Census Tract 2; Champaign County; Illinois,Champaign County,Illinois,2019,2023,4,1,ACS 2019 5-Year,ACS5,5-Year,21385.0,4938.0,701.0,273.0,428.0,38.94,61.06,428.0,7.0,62.0,25.0,141.0,94.0,54.91,32.94,21.96,922.0,701.0,221.0,76.03,23.97,4.1,18.2,922.0,564.0,16.0,56.0,0.0,1871.0,1812.0,255.0,1403.0,59.0,13.63,74.99,3.15,1,1,1,"{'has_income_metric': 1, 'median_household_inc...","{'occupied_units': 701, 'has_tenure_metric': 1...","{'rent_30_34': 7, 'rent_35_39': 62, 'rent_40_4...","{'for_rent_units': 4.1, 'other_vacant_units': ...","{'s1101_c01_001e': 701, 's1101_c01_001m': 92, ...","{'s1701_c01_001e': 1865, 's1701_c01_001m': 315...","{'s1901_c01_001e': 701, 's1901_c01_001m': 92, ...","{'s2301_c01_001e': 1414, 's2301_c01_001m': 248...","{'s1501_c01_001e': 325, 's1501_c01_001m': 127,...","{'s2401_c01_001e': 718, 's2401_c01_001m': 168,...","{'b03002_001e': 1871, 'b03002_001m': 318, 'b03...","{'s0101_c01_001e': 1871, 's0101_c01_001m': 318...","{'s1601_c01_001e': 1699, 's1601_c01_001m': 294..."
1,2019,17019000301,1400000US17019000301,17,019,000301,3.01,Census Tract 3.01,Census Tract 3.01; Champaign County; Illinois,Champaign County,Illinois,2019,2023,4,1,ACS 2019 5-Year,ACS5,5-Year,7099.0,1978.0,2055.0,0.0,2055.0,0.00,100.00,2055.0,17.0,53.0,64.0,1157.0,591.0,62.82,56.30,28.76,2475.0,2055.0,420.0,83.03,16.97,NaN,10.6,2475.0,52.0,0.0,0.0,68.0,5260.0,5073.0,3114.0,407.0,187.0,59.20,7.74,3.56,1,1,1,"{'has_income_metric': 1, 'median_household_inc...","{'occupied_units': 2055, 'has_tenure_metric': ...","{'rent_30_34': 17, 'rent_35_39': 53, 'rent_40_...","{'for_rent_units': None, 'other_vacant_units':...","{'s1101_c01_001e': 2055, 's1101_c01_001m': 191...","{'s1701_c01_001e': 5096, 's1701_c01_001m': 617...","{'s1901_c01_001e': 2055, 's1901_c01_001m': 191...","{'s2301_c01_001e': 5260, 's2301_c01_001m': 617...","{'s1501_c01_001e': 4769, 's1501_c01_001m': 663...","{'s2401_c01_001e': 2293, 's2401_c01_001m': 476...","{'b03002_001e': 5260, 'b03002_001m': 617, 'b03...","{'s0101_c01_001e': 5260, 's0101_c01_001m': 617...","{'s1601_c01_001e': 5260, 's1601_c01_001m': 617..."
2,2019,17019000302,1400000US17019000302,17,019,000302,3.02,Census Tract 3.02,Census Tract 3.02; Champaign County; Illinois,Champaign County,Illinois,2019,2023,4,1,ACS 2019 5-Year,ACS5,5-Year,10385.0,4237.0,1530.0,0.0,1530.0,0.00,100.00,1530.0,93.0,50.0,47.0,603.0,425.0,51.83,39.41,27.78,2065.0,1530.0,535.0,74.09,25.91,NaN,16.1,2065.0,44.0,16.0,21.0,60.0,2699.0,2558.0,948.0,66.0,141.0,35.12,2.45,5.22,1,1,1,"{'has_income_metric': 1, 'median_household_inc...","{'occupied_units': 1530, 'has_tenure_metric': ...","{'rent_30_34': 93, 'rent_35_3

,selected_metric_label,column_name
0,median_household_income,median_household_income
1,rent_burden,pct_rent_burden_30_plus
2,owner_occupied_share,pct_owner_occupied
3,auto_metric_01,not_hispanic_total
4,auto_metric_02,pct_white_non_hispanic
5,auto_metric_03,white_non_hispanic_population
6,auto_metric_04,occupied_housing_units_dp04
7,auto_metric_05,occupied_units
8,auto_metric_06,total_population
9,auto_metric_07,for_sale_only_units


## 5. Export datasets and summary tables

In [7]:

export_base_datasets(df, PATHS)

eda_run_summary = build_run_summary(df, numeric_columns, metric_map)
year_counts = year_tract_counts(df)
numeric_summary = numeric_summary_all_years(df, numeric_columns)
missingness = missingness_summary(df)

save_csv(eda_run_summary, PATHS["summary"] / "eda_run_summary.csv")
save_csv(year_counts, PATHS["summary"] / "year_tract_counts.csv")
save_csv(numeric_summary, PATHS["summary"] / "numeric_summary_by_year.csv")
save_csv(missingness, PATHS["summary"] / "missingness_by_year.csv")
save_metric_inventory(metric_map, numeric_columns, PATHS)

display(eda_run_summary)
display(year_counts)

,dataset,row_count,year_count,years,tract_count_union,stable_tract_count,numeric_column_count,selected_metric_count,selected_metrics
0,fact_acs_tract_profile,187,4,"2019, 2021, 2022, 2023",53,38,42,14,median_household_income:median_household_incom...


,year,tract_rows,distinct_tracts,stable_tract_rows
0,2019,43,43,38
1,2021,48,48,38
2,2022,48,48,38
3,2023,48,48,38


## 6. Correlations, rankings, distributions, trends, and heatmaps

In [8]:

save_correlations(df, corr_columns, PATHS)
save_rankings(df, metric_map, PATHS)
plot_metric_distributions(df, metric_map, PATHS)
plot_metric_trends(df, metric_map, PATHS)
build_tract_heatmaps(df, metric_map, PATHS, stable_only=False)
build_tract_heatmaps(df, metric_map, PATHS, stable_only=True)

print("[OK] All-years ACS EDA completed.")
print("Outputs saved under:", OUTPUT_ROOT)

[OK] All-years ACS EDA completed.
Outputs saved under: D:\Projects\Community-Pulse\outputs\acs\analysis\eda


## 7. Quick preview of key output files

In [9]:

summary_files = sorted((PATHS["summary"]).glob("*.csv"))
plot_corr_files = sorted((PATHS["plots_corr"]).glob("*.png"))
plot_trend_files = sorted((PATHS["plots_trend"]).glob("*.png"))
plot_dist_files = sorted((PATHS["plots_dist"]).glob("*.png"))
plot_heat_files = sorted((PATHS["plots_heat"]).glob("*.png"))

print("Summary files:")
for p in summary_files[:20]:
    print(" -", p.name)

print("\nCorrelation heatmaps:")
for p in plot_corr_files[:10]:
    print(" -", p.name)

print("\nTrend plots:")
for p in plot_trend_files[:10]:
    print(" -", p.name)

print("\nDistribution plots:")
for p in plot_dist_files[:10]:
    print(" -", p.name)

print("\nTract heatmaps:")
for p in plot_heat_files[:10]:
    print(" -", p.name)

Summary files:
 - eda_run_summary.csv
 - metric_inventory.csv
 - missingness_by_year.csv
 - numeric_summary_by_year.csv
 - tract_metric_heatmap_full.csv
 - tract_metric_heatmap_stable.csv
 - trend_auto_metric_01.csv
 - trend_auto_metric_02.csv
 - trend_auto_metric_03.csv
 - trend_auto_metric_04.csv
 - trend_auto_metric_05.csv
 - trend_auto_metric_06.csv
 - trend_auto_metric_07.csv
 - trend_auto_metric_08.csv
 - trend_auto_metric_09.csv
 - trend_auto_metric_10.csv
 - trend_auto_metric_11.csv
 - trend_median_household_income.csv
 - trend_owner_occupied_share.csv
 - trend_rent_burden.csv

Correlation heatmaps:
 - correlation_heatmap_2019.png
 - correlation_heatmap_2021.png
 - correlation_heatmap_2022.png
 - correlation_heatmap_2023.png

Trend plots:
 - trend_auto_metric_01.png
 - trend_auto_metric_02.png
 - trend_auto_metric_03.png
 - trend_auto_metric_04.png
 - trend_auto_metric_05.png
 - trend_auto_metric_06.png
 - trend_auto_metric_07.png
 - trend_auto_metric_08.png
 - trend_auto_metri